
# Experiment B: Recursive Character Text Splitting

## Objective
Comparing RecursiveCharacterTextSplitter with the original I used previously CharacterTextSplitter.

## Hypothesis
Recursive splitting may preserve paragraphs and sentences more naturally
by trying different separators before splitting at smaller boundaries.

## Controls
- Same source document
- Same embedding model
- Same chunk size and overlap where applicable
- Same test questions
- Separate chunk sets and vector-store collections

## Evaluation
- Number of chunks
- Chunk readability and boundaries
- Relevant information retrieved in the top results
- Whether the retrieved context supports the final answer


## INDEXING: DOCUMENT LOADER WITH PYPDF LOADER

In [140]:
from langchain_community.document_loaders import PyPDFLoader
import copy
import os
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters.character import CharacterTextSplitter

In [141]:
#instance of the docx loader
docxx_loader = Docx2txtLoader("Downloads/Introduction_to_Data_and_Data_Science2.docx")

In [142]:
pagee_docx = docxx_loader.load()

In [143]:
page_docxx_cut = copy.deepcopy(pagee_docx)
pagee_docx

[Document(metadata={'source': 'Downloads/Introduction_to_Data_and_Data_Science2.docx'}, page_content="Analysis vs Analytics\n\nAlright! So…\nLet’s discuss the not-so-obvious differences\nbetween the terms analysis and analytics.\nDue to the similarity of the words, some people\nbelieve they share the same meaning, and thus\nuse them interchangeably. Technically, this\nisn’t correct. There is, in fact, a distinct\ndifference between the two. And the reason\nfor one often being used instead of the other\nis the lack of a transparent understanding\nof both.\nSo, let’s clear this up, shall we?\nFirst, we will start with analysis.\nConsider the following…\nYou have a huge dataset containing data of\nvarious types. Instead of tackling the entire\ndataset and running the risk of becoming overwhelmed,\nyou separate it into easier to digest chunks\nand study them individually and examine how\nthey relate to other parts. And that’s analysis\nin a nutshell.\nOne important thing to remember, howev

In [144]:
for i in page_docxx_cut:
    i.page_content = " ".join(i.page_content.split())

In [145]:
len(page_docxx_cut[0].page_content)

8259

In [146]:
print("Number of loaded documents:", len("source"))

#for i, doc in enumerate("source"[:2]):
    #print(f"\n--- source {i + 1} ---")
    #print("Metadata:", page_docxx_cut.metadata)
    ##print("Extracted text:")
    #print(doc.page_content[:1500])

Number of loaded documents: 6


In [147]:
print(page_docxx_cut[0].page_content)

Analysis vs Analytics Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis. Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and running the risk of becoming overwhelmed, you separate it into easier to digest chunks and study them individually and examine how they relate to other parts. And that’s analysis in a nutshell. One important thing to remember, however, is that you perform analyses on things that have already happened in the past. Such as using an analysis to explain how a 

## INDEXING: DOCUMENT SPLITTING WITH Recursive TEXT SPLITTER

In [148]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [149]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    add_start_index=True
)

recursive_chunks = recursive_splitter.split_documents(page_docxx_cut)

In [150]:
len(recursive_chunks)

19

In [151]:
print("Number of recursive chunks:", len(recursive_chunks))

for i, chunk in enumerate(recursive_chunks[:5]):
    print(f"\n--- Chunk {i + 1} ---")
    print("Metadata:", chunk.metadata)
    print(chunk.page_content)

Number of recursive chunks: 19

--- Chunk 1 ---
Metadata: {'source': 'Downloads/Introduction_to_Data_and_Data_Science2.docx', 'start_index': 0}
Analysis vs Analytics Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will

--- Chunk 2 ---
Metadata: {'source': 'Downloads/Introduction_to_Data_and_Data_Science2.docx', 'start_index': 446}
So, let’s clear this up, shall we? First, we will start with analysis. Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and running the risk of becoming overwhelmed, you sep

In [152]:
len(recursive_chunks[17].page_content)

498

In [153]:
#chunk lenghts
for i, chunk in enumerate(recursive_chunks):
    print(
        f"Chunk {i + 1}: "
        f"{len(chunk.page_content)} characters"
    )

Chunk 1: 495 characters
Chunk 2: 497 characters
Chunk 3: 497 characters
Chunk 4: 496 characters
Chunk 5: 495 characters
Chunk 6: 497 characters
Chunk 7: 496 characters
Chunk 8: 496 characters
Chunk 9: 497 characters
Chunk 10: 498 characters
Chunk 11: 498 characters
Chunk 12: 495 characters
Chunk 13: 497 characters
Chunk 14: 488 characters
Chunk 15: 499 characters
Chunk 16: 493 characters
Chunk 17: 499 characters
Chunk 18: 498 characters
Chunk 19: 150 characters


In [154]:
chunk_lengths = [
    len(chunk.page_content)
    for chunk in recursive_chunks
]

print("Number of chunks:", len(chunk_lengths))
print("Shortest chunk:", min(chunk_lengths))
print("Longest chunk:", max(chunk_lengths))
print("Average chunk length:", round(
    sum(chunk_lengths) / len(chunk_lengths), 2
))

Number of chunks: 19
Shortest chunk: 150
Longest chunk: 499
Average chunk length: 477.95


## Initial Observations

- The recursive splitter produced 19 chunks.
- Chunk lengths ranged from 150 to 499 characters.
- The average chunk length was 477.95 characters.
- Most chunks were close to the configured 500-character target.
- The final chunk was shorter because it contained the remaining text.
- Further comparison with CharacterTextSplitter is needed to evaluate chunk quality.


## Indexing: Creating A Chroma Vector Store

In [155]:
%load_ext dotenv
%dotenv

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [156]:
import os
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [157]:
embedding = OpenAIEmbeddings(
    model="text-embedding-ada-002",
    api_key=os.getenv("OPEENAI_API_KEY")
)

In [158]:
# Vector / chroma
recursive_vectorstore = Chroma.from_documents(
    documents=recursive_chunks,
    embedding=embedding,
    collection_name="rag_recursive_experiment_b"
)

In [159]:
print(
    "Stored chunks:",
    recursive_vectorstore._collection.count()
)

Stored chunks: 38


## Retrieval: Similarity Search

In [ ]:
question = "What is the difference between data analysis and data analytics?"

retrieved_docs = recursive_vectorstore.similarity_search(
    question,
    k=5
)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print("Metadata:", doc.metadata)
    print(doc.page_content)


## Retrieval Experiment: Recursive Splitter

### Observations
- Similarity search retrieved relevant passages for the analysis-versus-analytics question.
- Some retrieved chunks began or ended mid-thought.
- Increasing k from 3 to 5 returned more context, but additional results were not necessarily more relevant.
- Source metadata and start indices were preserved in the retrieved chunks.

### Initial Conclusion
The recursive splitter supports useful semantic retrieval, but chunk relevance and completeness still need evaluation. More retrieved chunks do not automatically mean better answers.

### Next Step
Comparing similarity search with MMR using the same questions and the same recursive chunk collection.


## Retrieval: Maximal Marginal Relevance Search (MMR)

In [ ]:
question = "What is the difference between data analysis and data analytics?"

mmr_docs = recursive_vectorstore.max_marginal_relevance_search(
    question,
    k=3,
    fetch_k=10,
    lambda_mult=0.5
)

for i, doc in enumerate(mmr_docs, start=1):
    print(f"\n--- MMR Chunk {i} ---")
    print("Metadata:", doc.metadata)
    print(doc.page_content)

In [ ]:
mmr_relevance = recursive_vectorstore.max_marginal_relevance_search(
    question,
    k=3,
    fetch_k=10,
    lambda_mult=0.8
)

for i, doc in enumerate(mmr_relevance, start=1):
    print(f"\n--- Relevance-focused MMR {i} ---")
    print(doc.page_content)

In [ ]:
mmr_diversity = recursive_vectorstore.max_marginal_relevance_search(
    question,
    k=3,
    fetch_k=10,
    lambda_mult=0.2
)

for i, doc in enumerate(mmr_diversity, start=1):
    print(f"\n--- Diversity-focused MMR {i} ---")
    print(doc.page_content)


## Experiment C: Similarity Search vs. MMR

### Objective
Comparing retrieval relevance and diversity using the same question and document collection.

### Observations
- Similarity search returned relevant passages, but some context was incomplete.
- MMR with lambda_mult=0.8 returned passages closely related to the question.
- MMR with lambda_mult=0.2 returned more varied passages, including some less directly relevant content.
- Lower lambda values encouraged diversity but did not guarantee better answers.

### Initial Conclusion
The preferred retrieval strategy depends on the question and the desired balance between relevance and diversity. More testing is required before choosing a default.

### Limitations
These observations are based on a small number of example queries, not a formal retrieval benchmark.


## Generation: stuffing Documents

In [ ]:
#Constructing retriever chain
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.runnables.passthrough import RunnablePassthrough
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser

In [ ]:
#embedding
embedding = OpenAIEmbeddings(model = "text-embedding-ada-002",
                            api_key=os.getenv("OPEENAI_API_KEY"))

vectorstore_from_directory = Chroma( collection_name="rag_recursive_experiment_b",
                                     embedding_function= embedding,)

In [ ]:
len(recursive_vectorstore.get()["documents"])

In [ ]:
#Retriever
retriever = recursive_vectorstore.as_retriever(search_type = "mmr",
                                    search_kwargs = {"k": 3, 
                                                     "lambda_mult":0.8})

In [ ]:
#prompt template
TEMPLATE = """
Answer the following question:
{question}

Use only the information provided in the following context to answer the question:
{context}

If the context does not contain enough information to answer the question,
clearly state that you do not have enough information. Do not make up facts.
"""
prompt_template = PromptTemplate.from_template(TEMPLATE)

In [ ]:
#chat open ai
chaat = ChatOpenAI(model_name = "gpt-4.1",
                  api_key=os.getenv("OPEENAI_API_KEY"),
                  seed = 365,
                  temperature= 0,
                  max_tokens= 250)

In [ ]:
#input
Questions = "What is the difference between data analysis and data analytics?"

In [ ]:
chain = ({"context": retriever,
        "question": RunnablePassthrough()} 
         | prompt_template 
         |  chaat )

In [ ]:
chain.invoke(Questions)

## Genearation: Generating a Response

In [ ]:
chain = ({"context": retriever,
        "question": RunnablePassthrough()} 
         | prompt_template 
         |  chaat
         |  StrOutputParser())

In [ ]:
chain.invoke(Questions)


## End-to-End RAG Evaluation: Recursive Splitter

### Test question
What is the difference between data analysis and data analytics?

### Retrieval configuration
- Splitter: RecursiveCharacterTextSplitter
- Chunk size: 500 characters
- Chunk overlap: 50 characters
- Retrieval: MMR
- Retrieved chunks: 3

### Observation
The LLM explained data analysis using the retrieved context but stated that the context did not contain enough information to fully define data analytics.

### Interpretation
The answer reflected a limitation in the available context. This illustrates that a RAG system depends on both retrieval quality and the information contained in the retrieved passages.


## COMPARING CHARACTERSPLITTER WITH RECURSIVESPLIITER

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

# Recreating Experiment A using the same DOCX source
character_splitter = CharacterTextSplitter(
    separator=".",
    chunk_size=500,
    chunk_overlap=50
)

character_chunks = character_splitter.split_documents(page_docxx_cut)

print("Character chunks:", len(character_chunks))
print("Recursive chunks:", len(recursive_chunks))

In [ ]:
character_lengths = [
    len(chunk.page_content)
    for chunk in character_chunks
]

recursive_lengths = [
    len(chunk.page_content)
    for chunk in recursive_chunks
]

def summarize_chunks(name, lengths):
    print(f"\n{name}")
    print("-" * 30)
    print("Number of chunks:", len(lengths))
    print("Shortest:", min(lengths))
    print("Longest:", max(lengths))
    print("Average:", round(sum(lengths) / len(lengths), 2))

summarize_chunks("CharacterTextSplitter", character_lengths)
summarize_chunks("RecursiveCharacterTextSplitter", recursive_lengths)

# Short Observation on both comparison

- The recursive splitter is packing the text into chunks more efficiently in this experiment. 
- The character splitter uses the period as its separator, so sentence boundaries can leave chunks shorter than the target.
- The recursive splitter tries a hierarchy of separators, which can help it use the available space more effectively.


## Results: Initial Chunk-Length Comparison

| Metric | CharacterTextSplitter | RecursiveCharacterTextSplitter |
|---|---:|---:|
| Number of chunks | 21 |            19 |
| Shortest chunk | 184 |              150 |
| Longest chunk | 499 |               499 |
| Average chunk length | 400.00 |   477.95 |

### PS. Observations
- The recursive splitter produced two fewer chunks.
- Its average chunk length was closer to the 500-character target.
- Both methods produced chunks shorter than the target.
- The character splitter's period separator may contribute to shorter chunks.

### Preliminary conclusion
The recursive splitter used the available chunk capacity more efficiently in this test.\
However, chunk length alone does not establish superior retrieval or answer quality.

### Next step
Comparing retrieval results using identical questions, embedding models,\
and retrieval settings. Assess whether the retrieved chunks contain\
the information needed to answer each question.


## Retrieval: VectorStore - backed retrieval

In [ ]:
character_vectorstore = Chroma.from_documents(
    documents=character_chunks,
    embedding=embedding,
    collection_name="rag_character_experiment_a"
)

print(
    "Character store count:",
    character_vectorstore._collection.count()
)

print(
    "Recursive store count:",
    recursive_vectorstore._collection.count()
)

In [ ]:

# Check how many chunks are stored in each vector store

print(
    "Character store:",
     recursive_vectorstore._collection.count()
)

print(
    "Recursive store:",
    recursive_vectorstore._collection.count()
)


In [ ]:

question = "What is the difference between data analysis and data analytics?"

# Retrieve the top 3 chunks from each store
character_results = character_vectorstore .similarity_search(
    question,
    k=3
)

recursive_results = recursive_vectorstore.similarity_search(
    question,
    k=3
)

# Display character-based results
print("CHARACTER SPLITTER RESULTS")

for i, doc in enumerate(character_results, start=1):
    print(f"\n--- Result {i} ---")
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content)

# Display recursive results
print("\nRECURSIVE SPLITTER RESULTS")

for i, doc in enumerate(recursive_results, start=1):
    print(f"\n--- Result {i} ---")
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content)



## Experiment Results: Retrieval Comparison

### Test question
What is the difference between data analysis and data analytics?

### Retrieval settings
- Same DOCX source
- Same embedding model
- Similarity search
- Top 3 results (k=3)

### Observations
1. Both splitters retrieved the opening passage about analysis and analytics.
2. The remaining retrieved passages differed between the two approaches.
3. Some chunks contained incomplete explanations or began and ended mid-topic.
4. Both approaches retrieved relevant information,
5. but the completeness of the combined context varied.

### Preliminary conclusion
The splitting strategy influenced the passages retrieved for the question.\
However, these observations alone do not establish which method provides better retrieval quality.

### Next step
Comparring the final answers produced using each store,then assess their completeness and\
whether their claims are supported by the retrieved context.


## Generating Response comparing both answers character splitter and recursive splitter vectors

In [ ]:

character_retriever = character_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

recursive_retriever = recursive_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Both retrievers are ready.")


In [ ]:

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

TEMPLATE = """
Answer the question using only the supplied context.

If the context does not contain enough information,
say so clearly. Do not invent facts.

Question:
{question}

Context:
{context}
"""

prompt_template = PromptTemplate.from_template(TEMPLATE)

answer_chain = prompt_template | chaat | StrOutputParser()

In [ ]:

question = "What is the difference between data analysis and data analytics?"

# Experiment A: Character-based chunks
character_docs = character_retriever.invoke(question)

character_context = "\n\n".join(
    doc.page_content for doc in character_docs
)

character_answer = answer_chain.invoke({
    "question": question,
    "context": character_context
})

# Experiment B: Recursive chunks
recursive_docs = recursive_retriever.invoke(question)

recursive_context = "\n\n".join(
    doc.page_content for doc in recursive_docs
)

recursive_answer = answer_chain.invoke({
    "question": question,
    "context": recursive_context
})

print("=== CHARACTER SPLITTER ANSWER ===\n")
print(character_answer)

print("\n=== RECURSIVE SPLITTER ANSWER ===\n")
print(recursive_answer)



## Experiment B: CharacterTextSplitter vs RecursiveCharacterTextSplitter

### Objective
Compared how two text-splitting strategies affect chunk
statistics, retrieved context, and the final RAG answer.

### Chunk statistics

| Metric | CharacterTextSplitter | RecursiveCharacterTextSplitter |
|---|---:|---:|
| Number of chunks | 21 | 19 |
| Shortest chunk | 184 | 150 |
| Longest chunk | 499 | 499 |
| Average chunk length | 400.00 | 477.95 |

### Question
What is the difference between data analysis and data analytics?

### Observations
- The CharacterTextSplitter-based pipeline produced a\
  more complete explanation in this run.
- The RecursiveCharacterTextSplitter-based pipeline\
  gave a more cautious answer and identified limitations\
  in the retrieved context.
- The outputs suggest that chunking and retrieval may\
  affect the completeness of generated answers.

### Limitations
This comparison used one question and one observed\
answer from each pipeline. It does not establish that\
either splitter is generally superior.

### Conclusion
The experiment demonstrated that different splitting\
strategies can lead to different RAG answers. Further\
testing with multiple questions and consistent retrieval\
settings is needed to determine which strategy performs\
better for this document.


In [ ]:

question = "What is the difference between data analysis and data analytics?"

# A: Similarity search
similarity_docs = recursive_vectorstore.similarity_search(
    question,
    k=3
)

# B: MMR retrieval
mmr_docs = recursive_vectorstore.max_marginal_relevance_search(
    question,
    k=3,
    fetch_k=10,
    lambda_mult=0.5
)

print("=== SIMILARITY SEARCH ===\n")
for i, doc in enumerate(similarity_docs, start=1):
    print(f"--- Chunk {i} ---")
    print("Metadata:", doc.metadata)
    print(doc.page_content)
    print()

print("\n=== MMR RETRIEVAL ===\n")
for i, doc in enumerate(mmr_docs, start=1):
    print(f"--- Chunk {i} ---")
    print("Metadata:", doc.metadata)
    print(doc.page_content)
    print()


# Experiment B: Evaluating Text Splitting and Retrieval in a RAG Pipeline

## 1. Overview

This experiment explored how different document-splitting strategies and retrieval settings influence the behaviour of a Retrieval-Augmented Generation (RAG) pipeline built with Python and LangChain.

The goal was not simply to generate an answer, but to understand how document preparation and retrieval affect the information supplied to a language model and, ultimately, the response it generates.

## 2. Objectives

- Compare `CharacterTextSplitter` and `RecursiveCharacterTextSplitter`.
- Measure and compare the resulting chunk lengths.
- Store the recursive chunks in a Chroma vector store.
- Inspect retrieved passages using similarity search.
- Explore Maximum Marginal Relevance (MMR) with different relevance-diversity settings.
- Compare the final LLM answers produced by the two splitting pipelines.

## 3. Experimental Setup

The experiment used a DOCX document titled `Introduction_to_Data_and_Data_Science2.docx`.

The workflow included:

1. Loading the document using LangChain's document loader.
2. Preprocessing the extracted text.
3. Splitting the document into chunks.
4. Generating embeddings and storing chunks in Chroma.
5. Retrieving relevant chunks for a question.
6. Passing the retrieved context to a language model through a prompt template.
7. Comparing the generated responses.

The test question was:

**“What is the difference between data analysis and data analytics?”**

## 4. Chunk Statistics

The two splitting strategies produced the following results:

| Metric | CharacterTextSplitter | RecursiveCharacterTextSplitter |
|---|---:|---:|
| Number of chunks | 21 | 19 |
| Shortest chunk | 184 characters | 150 characters |
| Longest chunk | 499 characters | 499 characters |
| Average chunk length | 400.00 characters | 477.95 characters |

### Interpretation

The recursive splitter produced two fewer chunks and a higher average chunk length in this experiment.

This shows that the two splitting strategies partitioned the document differently. However, chunk count and length alone cannot determine which strategy produces better retrieval results.

## 5. Vector Store Creation

The recursive chunks were embedded and stored in a dedicated Chroma collection named `rag_recursive_experiment_b`.

The collection count returned **19 stored chunks**, matching the number of recursive chunks created.

This confirmed that the expected number of chunks had been stored in the experiment's vector collection.

## 6. Retrieval Experiments

### Similarity Search

Similarity search was used to retrieve passages related to the test question.

Inspecting the retrieved passages helped reveal which parts of the original document were supplied to the language model.

The results demonstrated that retrieval quality is important: the model can only use the retrieved context available to it when following a context-restricted prompt.

### Maximum Marginal Relevance (MMR)

MMR was explored to investigate the balance between relevance and diversity among retrieved passages.

The experiments included different `lambda_mult` values:

- `0.8`: places greater emphasis on relevance.
- `0.5`: balances relevance and diversity.
- `0.2`: places greater emphasis on diversity.

The retrieved passages changed as the setting changed. Some results remained similar, while others brought in different sections of the document.

This illustrated how retrieval configuration can influence the context supplied to the LLM. It did not establish that one setting was universally best.

## 7. Comparing the Generated Answers

Both splitting pipelines were tested with the same question.

### CharacterTextSplitter result

The answer provided a relatively complete explanation. It described analysis as focusing on understanding past events and analytics as using reasoning and patterns to explore possible future outcomes.

### RecursiveCharacterTextSplitter result

The answer explained data analysis but was more cautious about defining data analytics. It stated that the retrieved context did not contain enough information to fully explain the distinction.

### Interpretation

In this particular run, the CharacterTextSplitter pipeline produced the more complete answer, while the recursive pipeline produced a more cautious response.

One possible explanation is that the two pipelines retrieved different combinations of relevant passages. The prompt's instruction to acknowledge insufficient information may also have contributed to the difference.

This is an observation from one question and one comparison, not proof that CharacterTextSplitter is generally superior or that RecursiveCharacterTextSplitter is inherently more accurate.

## 8. Key Lessons Learned

1. **Chunking matters.** Different splitting strategies can change the boundaries and sizes of the text passages available for retrieval.
2. **Retrieval matters.** Relevant information must reach the LLM for it to produce a well-supported answer.
3. **MMR offers a retrieval trade-off.** Its configuration influences how the system balances relevance and diversity.
4. **More context is not automatically better.** The usefulness of retrieved passages matters more than simply increasing their number.
5. **An LLM can acknowledge missing evidence.** Prompt instructions can encourage cautious answers, although they cannot guarantee that every response will be correct.
6. **Evaluation is essential.** A successful execution does not by itself demonstrate that a RAG pipeline retrieves the right information or generates a reliable answer.

## 9. Limitations

- The splitter comparison used one main test question.
- The observed answers do not establish general performance across different questions.
- The retrieved passages and final responses were inspected qualitatively rather than evaluated with a formal retrieval metric.
- Differences in retrieved context, prompt behaviour, and model generation may all have contributed to the answer comparison.

## 10. Conclusion

This experiment provided practical insight into how document splitting, vector storage, retrieval configuration, and LLM generation work together in a RAG pipeline.

The most important finding was that changing the splitting strategy coincided with a difference in the completeness and caution of the generated answer.

The next stage is to evaluate both pipelines using several questions, consistent settings, and explicit criteria for retrieval relevance and answer support.

**Overall takeaway:** Building a RAG system is not just about connecting a retriever to an LLM. It also requires inspecting evidence, comparing design choices, and evaluating the quality of the resulting answers.


# Experiment C: Comparing Similarity Search and Maximum Marginal Relevance (MMR) in RAG

## 1. Objective

The objective of this experiment was to compare two retrieval strategies used in Retrieval-Augmented Generation (RAG): similarity search and Maximum Marginal Relevance (MMR).

I wanted to understand how each method retrieves information from a Chroma vector store and how the retrieved context affects the quality of the final answer generated by a language model.

## 2. Experimental Setup

I used the same document collection, embedding setup, and question to compare the retrieval methods.

- **Vector database:** Chroma
- **Embedding model:** `text-embedding-ada-002`
- **Retrieval methods:** Similarity search and MMR
- **Number of retrieved chunks:** 3 for the comparison
- **Language model:** OpenAI chat model
- **Evaluation approach:** Inspecting retrieved context and comparing generated answers

For MMR, I also experimented with different `lambda_mult` values to observe the balance between relevance and diversity.

## 3. Observations

| Evaluation criterion | Similarity search | MMR |
|---|---|---|
| Context sufficiency | Yes | Partial |
| Repetition in retrieved context | No | No |

Similarity search returned context that appeared sufficient to answer the selected question. MMR also retrieved useful information, but the context was only partially sufficient for the same question.

Neither method showed obvious repetition in the retrieved chunks during this test.

## 4. Interpreting the Results

### Similarity search

Similarity search retrieves documents based on how closely their embeddings match the query embedding.

In this experiment, the retrieved context provided enough information for the language model to produce a more complete answer to the question.

### Maximum Marginal Relevance (MMR)

MMR considers both relevance to the query and diversity among the retrieved documents. Its goal is to reduce redundant results while still retrieving useful information.

In my experiment, MMR returned relevant information without obvious repetition, but the context did not contain everything needed to answer the question fully.

This illustrates an important trade-off: diversity can be useful, but retrieved chunks must still provide sufficient information to answer the question.

## 5. Conclusion

For this particular question and document collection, similarity search performed better in terms of context sufficiency, while both methods avoided obvious repetition.

However, this result does not establish that similarity search is always better than MMR. Performance depends on the question, document structure, chunking strategy, retrieval parameters, and evaluation criteria.

The experiment helped me understand that choosing a retrieval strategy involves more than retrieving semantically similar text. The goal is to retrieve context that is both relevant and sufficient for the language model to generate a grounded answer.

## 6. What I Learned

- Similarity search prioritizes semantic relevance to the query.
- MMR balances relevance with diversity among retrieved results.
- Relevant chunks are not necessarily sufficient to answer a question completely.
- Retrieval quality directly affects the context available to the generation stage.
- Comparing actual retrieved chunks and generated answers is more informative than relying on assumptions about which retrieval method should perform better.

## 7. Next Steps

Further experiments could compare different questions, values of `k`, and MMR `lambda_mult` settings. I could also evaluate whether the correct supporting information appears in the retrieved top three chunks and whether the final answer is fully supported by that context.

This experiment is part of my ongoing journey toward understanding and building practical RAG systems with Python, LangChain, embeddings, vector databases, and large language models.